# 📓 Semana 22 · Dia 2 — Gravação no Bronze e disparo automático do DLT

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | Portfólio empresarial |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Bronze + DLT automático |

---


## 📖 Teoria — Do app ao Lakehouse

Aprovado → as linhas viram uma **tabela Bronze** (com `fluxo_id`, `submissao_id`) → um **job/pipeline DLT** detecta a nova submissão e atualiza Prata/Ouro.

Assim, o app alimenta o Lakehouse com governança e reprocessamento.


### 💻 Na prática — Gravando no Bronze

Grave a submissão aprovada com metadata.


In [ ]:
# Gravar submissão aprovada no Bronze
def gravar_bronze(sid, df, fluxo_id):
    tabela = f"workspace.bronze.{fluxo_id}_bronze"
    spark.sql(f"CREATE TABLE IF NOT EXISTS {tabela} (
        submissao_id STRING, linha INT, dados STRING,
        _ingested_at TIMESTAMP) USING DELTA")
    registros = [(sid, i, row.to_json(), "now")
                 for i, row in df.iterrows()]
    spark.createDataFrame(registros, ["submissao_id", "linha", "dados", "_ingested_at"])\
        .withColumn("_ingested_at", current_timestamp())\
        .write.mode("append").saveAsTable(tabela)
    print(f"{len(registros)} linhas no Bronze {fluxo_id}.")
print("gravar_bronze pronto (append-only com metadata).")

In [ ]:
# Disparar o pipeline (via API de jobs)
def disparar_dlt():
    print("""
    1. Job DLT que lê workspace.bronze.*_bronze
    2. Pós-gravação: POST /api/2.1/jobs/run-now {job_id: X}
    3. O DLT aplica expectations e atualiza Prata/Ouro
    """)
    print("Disparo automático: app → job → DLT → Prata/Ouro.")
disparar_dlt()

### 💻 Na prática — Pipeline DLT do fluxo

O pipeline lê o Bronze do fluxo e aplica qualidade.


In [ ]:
# ===== workspace_file: pipeline_fluxos.py =====
import dlt
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StringType, StructType

@dlt.table
@dlt.expect_all_or_drop({"submissao_presente": "submissao_id IS NOT NULL"})
def fluxos_prata():
    return (spark.readStream
        .format("delta")
        .table("workspace.bronze.metas_bronze"))
print("Pipeline DLT dos fluxos (leia o Bronze, valide, atualize Prata).")

> 🎯 **Dica de prova**: Portfólio: app → Bronze → DLT → Prata é o ciclo de governança completo — o que o curso inteiro ensinou aplicado ao produto.


## 🎯 Exercícios de fixação

**1.** Por que o Bronze guarda submissao_id?

**2.** O que o DLT faz com linhas que violam expectations?

**3.** Simule: aprove, grave, rode o DLT e confira a Prata.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** submissao_id

Rastreabilidade de ponta a ponta: da submissão à Prata (linhagem).

**2.** Expectations

expect_or_drop descarta; expect_or_fail falha — conforme a regra.

**3.** Simular

Fluxo completo: submissão → aprovação → Bronze → DLT → Prata.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*